# Closing three gaps left open by the n=300 run

Fourth notebook. `03_scaleup_n300.ipynb` produced the reference result:
reading handwriting works (AUROC 0.835), judging mistakes does not (0.522,
because the model grades at 51.7% against a 0.500 baseline). This notebook runs
the three follow-ups that needed new generation, in one session.

**A. Token-confidence baseline (transcription).** Entropy beats a plain count of
distinct answers, but that is a cruder version of the same idea, not an
independent signal. Nobody can currently say whether the model's own token
probabilities would do just as well. We never recorded them, so this captures
per-token log-probabilities alongside each sample.

**B. Grading prompt variants (screening).** The model answers "there is a
mistake" on 93% of items. That may be a capability ceiling, or it may be an
artifact of a yes/no question with no counter-pressure. Three variants against
the FERMAT baseline: restate-the-result-first, an explicit statement that half
the answers are correct, and one asking for a 0-100 confidence. Run on the
first 100 items only -- this is a screen for a large effect, not a measurement.
If a variant moves accuracy off 0.500, it earns a full 300-item run.

**C. Transcription at K=10.** At K=5 entropy takes only 7 distinct values, which
is why the deferral rule jumps from 13% coverage to 33% with nothing in between.
K=10 gives 11 levels. AUROC was also still rising with K (0.74 at three samples,
0.78 at four, 0.835 at five), so this likely improves the headline too.

**Same 300 items as the reference run**, drawn by the same seeded call, so
everything here joins back to `reference/n300_balanced_20260802.json`.

**Run order:** cells 1-4, then 5 (A+C, the long one), then 6 (B), then 7-9.
Everything is checkpointed per item and resumes after a disconnect.

In [9]:
# Set before anything touches CUDA. Capturing token log-probabilities keeps a
# large logit tensor per generated step, and the resulting allocation pattern
# fragments memory badly on a 16GB card. expandable_segments lets the allocator
# grow a segment instead of failing on a fit it technically has room for.
# Only takes effect on a fresh runtime -- if the model is already loaded, the
# batch-size backoff in the sampling cell is what saves the run instead.
import os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Install cell: GPU-dependent packages only.
# `datasets` is intentionally also in the local requirements.txt -- each
# environment installs its own copy independently, no conflict.
# torch is not installed explicitly: Colab GPU runtimes ship with it preinstalled.
# sentence-transformers is installed here, not in the optional NLI cell at
# the end, so pip resolves every version constraint ONCE before the model
# loads. Installing it mid-session could pull a different transformers
# version underneath an already-loaded model.
!pip install -q transformers accelerate bitsandbytes datasets qwen-vl-utils sentence-transformers

In [10]:
# Auth & code/results access cell.
import json
import os
from getpass import getpass

from huggingface_hub import login

# --- Drive mount first: it holds both the model cache and the token store ---
from google.colab import drive

drive.mount("/content/drive")
PROJECT_DIR = "/content/drive/MyDrive/uncertainty-math-vlm"
DRIVE_MODEL_CACHE = f"{PROJECT_DIR}/model_cache"
os.makedirs(DRIVE_MODEL_CACHE, exist_ok=True)

# --- Tokens: entered ONCE, then cached on your Drive ---
# Deliberately not hardcoded in this notebook. This file is tracked in a
# public repo, and GitHub's secret scanning auto-revokes any ghp_ token that
# lands in a public commit -- so an inline token would stop working by
# itself. Drive is private to your account, survives runtime recycling, and
# git never touches it, so you get the same "no retyping" result safely.
TOKEN_FILE = f"{PROJECT_DIR}/.tokens.json"
RESET_TOKENS = False  # set True once to replace previously saved tokens


def get_token(name, prompt):
    """Return a saved token, prompting (once) and persisting it if absent."""
    tokens = {}
    if os.path.exists(TOKEN_FILE):
        with open(TOKEN_FILE) as f:
            tokens = json.load(f)
    if RESET_TOKENS or not tokens.get(name):
        tokens[name] = getpass(prompt).strip()
        with open(TOKEN_FILE, "w") as f:
            json.dump(tokens, f)
        os.chmod(TOKEN_FILE, 0o600)
        print(f"Saved {name} to Drive -- you will not be asked for it again.")
    return tokens[name]


HF_TOKEN = get_token("HF_TOKEN", "Hugging Face token (asked once): ")
GH_TOKEN = get_token("GH_TOKEN", "GitHub token with 'repo' scope (asked once): ")

if not HF_TOKEN.startswith("hf_"):
    raise ValueError(
        "Stored Hugging Face token does not start with 'hf_'. Set "
        "RESET_TOKENS = True and re-run this cell to replace it."
    )

login(token=HF_TOKEN)
print("Hugging Face login OK")

# --- Clone the repo (code + results live in the same repo for this pilot) ---
# Cloned anonymously: the repo is public, so read access needs no token, and
# keeping the token out of the clone URL means a clone error can never echo
# it into this notebook's saved output. The token is used only to push.
REPO_URL = "https://github.com/sepehrmaleki369/uncertainty-math-vlm.git"

# Remove any stale clone from a previous (possibly failed) run so this cell
# is safe to re-run -- git clone silently no-ops into a pre-existing
# directory, which would otherwise leave `repo/` incomplete without error.
!rm -rf repo
!git clone -q {REPO_URL} repo

# %pip (not !pip) installs into the *running kernel's* environment -- !pip
# can silently target a different Python install.
%pip install -q -e repo/

# An editable install writes an `__editable__.pilot-*.pth` file into
# site-packages, but .pth files are only processed by the `site` module at
# INTERPRETER STARTUP. The kernel is already running, so it never sees them
# and `import pilot` fails with ModuleNotFoundError even though the install
# reported success. Putting the repo on sys.path directly makes the package
# importable right now, with no kernel restart needed.
import importlib
import sys

REPO_DIR = os.path.abspath("repo")
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
importlib.invalidate_caches()

import pilot.data
import pilot.prompts
import pilot.parsing
import pilot.entropy

print(f"pilot package imported from: {os.path.dirname(pilot.__file__)}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Hugging Face login OK
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for pilot (pyproject.toml) ... done
pilot package imported from: /content/repo/pilot


In [11]:
# Model load cell.
# Start with the 3B model for the first smoke test -- same prompt format and
# code path as the 7B, but noticeably faster to load and run, so early bugs
# get caught cheaply. Swap MODEL_ID to the 7B line below once the pipeline
# runs cleanly end to end on the 3B.
import torch
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"
# MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"  # swap in once the 3B pipeline is clean

# If memory is tight on the 7B, load in 4-bit instead:
# from transformers import BitsAndBytesConfig
# quantization_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_compute_dtype=torch.bfloat16,
# )
# model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
#     MODEL_ID,
#     quantization_config=quantization_config,
#     device_map="auto",
#     cache_dir=DRIVE_MODEL_CACHE,
# )

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    cache_dir=DRIVE_MODEL_CACHE,
)
processor = AutoProcessor.from_pretrained(MODEL_ID, cache_dir=DRIVE_MODEL_CACHE)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

In [12]:
# Sample cell. Must reproduce the reference run's items exactly, so the new
# measurements join back to it -- same function, same seed, same balance.
import logging

import pilot.data

logging.basicConfig(level=logging.WARNING, force=True)

N = 300
SEED = 42
TARGET_ERROR_FRAC = 0.5
N_SCREEN = 100   # prompt-variant screening subset (see intro)

sample = pilot.data.load_fermat_balanced(
    n=N, seed=SEED, target_error_frac=TARGET_ERROR_FRAC
)
N = len(sample)
n_error = sum(bool(x) for x in sample["has_error"])
print(f"{N} items, {n_error} with a mistake, {N - n_error} clean")
print(f"screening subset: first {N_SCREEN} items "
      f"({sum(bool(x) for x in sample['has_error'][:N_SCREEN])} with a mistake)")

README.md:   0%|          | 0.00/3.74k [00:00<?, ?B/s]

data/train-00000-of-00010.parquet: reconstructing file:   0%|          |  0.00B /  467MB            

data/train-00000-of-00010.parquet: downloading bytes:           |  0.00B            

data/train-00001-of-00010.parquet: reconstructing file:   0%|          |  0.00B /  481MB            

data/train-00001-of-00010.parquet: downloading bytes:           |  0.00B            

...

data/train-00008-of-00010.parquet: downloading bytes:           |  0.00B            

data/train-00009-of-00010.parquet: reconstructing file:   0%|          |  0.00B /  493MB            

data/train-00009-of-00010.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/2244 [00:00<?, ? examples/s]

300 items, 150 with a mistake, 150 clean
screening subset: first 100 items (49 with a mistake)


In [13]:
# A + C: five more transcription samples per item, capturing per-token
# log-probabilities this time.
#
# Reaching K=10 does two things at once: it doubles the entropy resolution
# (7 distinct values at K=5 -> 11 at K=10, which is what limits the deferral
# rule), and it gives the token-confidence baseline. Both come from the same
# generations, so this costs one pass, not two.
import gc
import json
import os
import time

import torch
from qwen_vl_utils import process_vision_info
from tqdm.auto import tqdm

import pilot.prompts

# Capturing per-token log-probabilities requires output_scores=True, which
# routes generation through transformers' output-capturing wrapper. That wrapper
# makes the *vision tower* materialize its full attention matrix instead of
# using the memory-efficient SDPA path, and for these images that is a constant
# 14.02 GiB request -- identical at batch 1 and batch 5, which is what proves it
# is not the sampling batch. It does not fit beside a 6 GiB model on a 16 GiB
# card, for any item. Notebook 03 worked precisely because it never asked for
# scores.
#
# So experiment A (the token-confidence baseline) needs a card with room for
# that 14 GiB on top of the ~6 GiB model. Measured: a 16 GiB T4 cannot, a 40 GiB
# A100 can with room to spare. B and C do not need log-probabilities at all and
# run fine on the plain path below regardless.
#
# Decided from the actual device rather than hardcoded, so moving to a bigger
# runtime just works and staying on a T4 does not silently waste an hour
# rediscovering the same OOM. Set the value directly to override.
_VRAM_GIB = (torch.cuda.get_device_properties(0).total_memory / 1024**3
             if torch.cuda.is_available() else 0.0)
_LOGPROB_VRAM_MIN_GIB = 24.0   # T4/V100 are 16 and fail; L4 is 24 and fits

CAPTURE_LOGPROBS = _VRAM_GIB >= _LOGPROB_VRAM_MIN_GIB

print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'} "
      f"({_VRAM_GIB:.1f} GiB)")
if CAPTURE_LOGPROBS:
    print("  enough memory for token-confidence capture: running experiments A, B and C")
    print("  note: output_scores forces eager attention (no SDPA/flash) through the")
    print("  whole model, including the vision tower. Attention there scales with the")
    print("  square of image patch count, so large images can take noticeably longer")
    print("  per item than the plain path. A per-item timer below shows progress so a")
    print("  slow item does not look like a hang.")
else:
    print(f"  under {_LOGPROB_VRAM_MIN_GIB:.0f} GiB, so experiment A (token confidence) is")
    print("  skipped; B and C run normally. An A100 runtime would enable A.")

K_EXTRA = 5          # new samples per item, on top of the reference run's 5
TEMP = 0.7

META_FIELDS = ("orig_q", "pert_a", "has_error", "handwriting_style", "image_quality")
INFRA_EXCEPTIONS = (ConnectionError, TimeoutError, torch.cuda.OutOfMemoryError, OSError)


# Batch size for logprob capture, with automatic backoff.
#
# output_scores=True retains a (batch x vocab) logit tensor for every generated
# step, which notebook 03 did not pay for. On a T4 that extra pressure makes the
# batch-5 prefill OOM even though 03 batched 5 successfully. So: start at 5, drop
# to 2, then 1, and STAY at whatever works. Retrying 5 on every item would spend
# a failed multi-second prefill per item for the rest of the run.
class ImageTooLargeForGPU(RuntimeError):
    """One item's image cannot be vision-encoded on this GPU at any batch size.

    Raised instead of letting the OOM propagate, so the loop can record the
    item and keep going. Losing a handful of items to a hardware limit is a
    reportable gap; losing the whole run to one of them is not.
    """


_BATCH_LADDER = [5, 2, 1]
_batch_state = {"index": 0}


def _generate_batch_plain(messages, n: int, temperature: float):
    """Generation without score capture -- byte-for-byte the call notebook 03
    used, which is known to run these images at batch 5 on a T4."""
    text_prompt = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text_prompt], images=image_inputs, videos=video_inputs,
        padding=True, return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs, max_new_tokens=512, do_sample=True,
            temperature=temperature, num_return_sequences=n,
        )

    trimmed = output_ids[:, inputs["input_ids"].shape[1]:]
    texts = processor.batch_decode(
        trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )
    del output_ids, inputs
    gc.collect()
    torch.cuda.empty_cache()
    return texts, [None] * len(texts), [None] * len(texts)


def _generate_batch_with_logprobs(messages, n: int, temperature: float):
    """One generate() call for n sequences, returning texts and per-sample
    log-probability summaries. Raises OutOfMemoryError for the caller to handle."""
    text_prompt = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text_prompt], images=image_inputs, videos=video_inputs,
        padding=True, return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=512, do_sample=True,
            temperature=temperature, num_return_sequences=n,
            return_dict_in_generate=True, output_scores=True,
        )

    prompt_len = inputs["input_ids"].shape[1]
    trimmed = out.sequences[:, prompt_len:]
    texts = processor.batch_decode(
        trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )
    scores = model.compute_transition_scores(
        out.sequences, out.scores, normalize_logits=True
    )
    means, mins = [], []
    for row in scores:
        finite = row[torch.isfinite(row)]
        if len(finite) == 0:
            means.append(None); mins.append(None); continue
        means.append(float(finite.mean()))
        mins.append(float(finite.min()))

    # Free the score tensors before the next item: they are the largest thing
    # held here, and leaving them to the garbage collector is what turns a
    # tight fit into an OOM two items later.
    del out, scores, inputs
    gc.collect()
    torch.cuda.empty_cache()
    return texts, means, mins


def generate_with_logprobs(messages, n: int, temperature: float):
    """Draw n samples, backing the batch size off on OOM rather than failing.

    compute_transition_scores gives the log-probability actually assigned to
    each chosen token. Two summaries per sample: the mean over tokens (overall
    fluency) and the minimum (the single least confident token, which is where
    a misread digit shows up).

    The scores are post-temperature-warping, so these are confidences under the
    sampling distribution rather than the raw model distribution. That is the
    right comparison here: entropy is computed over samples from that same
    warped distribution, so both signals see the same generations.
    """
    texts, means, mins = [], [], []
    while len(texts) < n:
        want = n - len(texts)
        size = min(_BATCH_LADDER[_batch_state["index"]], want)
        try:
            fn = (_generate_batch_with_logprobs if CAPTURE_LOGPROBS
                  else _generate_batch_plain)
            t, me, mi = fn(messages, size, temperature)
            texts += t; means += me; mins += mi
        except torch.cuda.OutOfMemoryError:
            gc.collect()
            torch.cuda.empty_cache()
            if _batch_state["index"] + 1 < len(_BATCH_LADDER):
                _batch_state["index"] += 1
                print(f"  OOM at batch {size}; dropping to "
                      f"{_BATCH_LADDER[_batch_state['index']]} for the rest of the run.",
                      flush=True)
                continue
            # Already at batch 1. The failure is then in the vision tower, not
            # the sampling batch: attention over image patches is quadratic in
            # patch count, so one unusually large scan can demand more memory
            # than the card has no matter how few sequences we ask for. Observed
            # asking for the same 14.02 GiB at batch 1 as at batch 5, which is
            # the tell that num_return_sequences was never the driver.
            raise ImageTooLargeForGPU(
                f"vision encoding did not fit even at batch 1"
            ) from None
    return texts, means, mins


CHECKPOINT_DIR = "/content/drive/MyDrive/uncertainty-math-vlm/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
extra_path = (f"{CHECKPOINT_DIR}/extra_transcription_k{K_EXTRA}_"
              f"{MODEL_ID.split('/')[-1]}_n{N}_seed{SEED}"
              f"{'_logprobs' if CAPTURE_LOGPROBS else ''}.jsonl")

skipped = []
extra_results = []
if os.path.exists(extra_path):
    with open(extra_path) as f:
        extra_results = [json.loads(line) for line in f if line.strip()]
    # Only trust a prefix that matches the current sample order.
    valid = []
    stale_skips_dropped = 0
    for idx, entry in enumerate(extra_results[:N]):
        item = sample[idx]
        if not all(entry["item"].get(k) == item[k] for k in META_FIELDS):
            print(f"Checkpoint item {idx + 1} does not match sample order; resuming there.")
            break
        skip_trustworthy = (
            entry.get("skipped_oom") and entry.get("skipped_capture_mode") == CAPTURE_LOGPROBS
        )
        stale_skip = entry.get("skipped_oom") and not skip_trustworthy
        if stale_skip:
            stale_skips_dropped += 1
        if not skip_trustworthy and len(entry.get("samples_raw", [])) != K_EXTRA:
            # Stop at the first entry this run cannot trust, exactly as the
            # sample-order check above does -- everything from here on gets
            # regenerated. Do NOT count later entries as stale here: they were
            # never inspected, so claiming a count for them would be a guess.
            break
        valid.append(entry)
    if len(valid) != len(extra_results):
        with open(extra_path, "w") as f:
            for e in valid:
                f.write(json.dumps(e, default=str) + "\n")
    extra_results = valid
    print(f"Resuming from {len(extra_results)} completed items")
    if stale_skips_dropped:
        print(f"  Stopped early: hit an item skipped under a different capture "
              f"mode (CAPTURE_LOGPROBS={not CAPTURE_LOGPROBS} then, "
              f"{CAPTURE_LOGPROBS} now). That item and everything after it in "
              f"the checkpoint will be regenerated under the current mode.")

if len(extra_results) >= N:
    print("All items already done.")
else:
    with tqdm(total=(N - len(extra_results)) * K_EXTRA, desc="transcription", unit="sample") as pbar:
        for item_idx, item in enumerate(sample):
            if item_idx < len(extra_results):
                continue
            messages = pilot.prompts.build_transcription_messages(item["image"])
            _t0 = time.time()
            try:
                texts, means, mins = generate_with_logprobs(messages, K_EXTRA, TEMP)
            except ImageTooLargeForGPU:
                print(f"  item {item_idx + 1}: image too large for this GPU; "
                      "recording as skipped and continuing.", flush=True)
                skipped.append(item_idx)
                texts, means, mins = [], [], []
            _elapsed = time.time() - _t0
            if CAPTURE_LOGPROBS or _elapsed > 20:
                print(f"  item {item_idx + 1}/{N}: {_elapsed:.1f}s"
                      f"{' (slow -- large image or eager attention, not stuck)' if _elapsed > 20 else ''}",
                      flush=True)
            entry = {
                "item": {k: item[k] for k in META_FIELDS},
                "samples_raw": texts,
                "mean_logprob": means,
                "min_logprob": mins,
                "elapsed_seconds": _elapsed,
                # Written so a resumed run does not retry a known-impossible
                # item forever, and so scoring can report the gap honestly.
                # Tagged with the mode it happened under: an item that OOM'd
                # while capturing logprobs may well succeed on the much
                # cheaper plain path, so a skip is only trustworthy if it
                # happened under the SAME mode the current run is using.
                "skipped_oom": len(texts) == 0,
                "skipped_capture_mode": CAPTURE_LOGPROBS if len(texts) == 0 else None,
            }
            extra_results.append(entry)
            with open(extra_path, "a") as f:
                f.write(json.dumps(entry, default=str) + "\n")
                f.flush()
            pbar.update(K_EXTRA)

n_skipped = sum(1 for e in extra_results if e.get("skipped_oom"))
print(f"extra_results: {len(extra_results)} items ({n_skipped} skipped for GPU memory)")
if n_skipped:
    print("  Those items keep their K=5 reference samples; they simply have no")
    print("  K=10 entropy or token-confidence value. Scoring reports the gap.")

GPU: NVIDIA A100-SXM4-40GB (39.5 GiB)
  enough memory for token-confidence capture: running experiments A, B and C
  note: output_scores forces eager attention (no SDPA/flash) through the
  whole model, including the vision tower. Attention there scales with the
  square of image patch count, so large images can take noticeably longer
  per item than the plain path. A per-item timer below shows progress so a
  slow item does not look like a hang.


transcription:   0%|          | 0/1500 [00:00<?, ?sample/s]

  item 1/300: 20.3s (slow -- large image or eager attention, not stuck)
  item 2/300: 13.0s
  item 3/300: 10.9s
  item 4/300: 12.7s
  item 5/300: 28.0s (slow -- large image or eager attention, not stuck)
  item 6/300: 12.9s
  item 7/300: 14.1s
  item 8/300: 14.3s
  item 9/300: 14.2s
  item 10/300: 19.3s
  item 11/300: 9.0s
  item 12/300: 18.3s
  item 13/300: 20.3s (slow -- large image or eager attention, not stuck)
  item 14/300: 21.2s (slow -- large image or eager attention, not stuck)
  item 15/300: 11.5s
  item 16/300: 12.6s
  item 17/300: 11.8s
  item 18/300: 17.3s
  item 19/300: 11.6s
  item 20/300: 26.3s (slow -- large image or eager attention, not stuck)
  item 21/300: 24.4s (slow -- large image or eager attention, not stuck)
  item 22/300: 13.4s
  item 23/300: 16.3s
  item 24/300: 18.2s
  item 25/300: 11.3s
  item 26/300: 16.7s
  item 27/300: 23.8s (slow -- large image or eager attention, not stuck)
  item 28/300: 11.5s
  item 29/300: 24.4s (slow -- large image or eager attenti

In [14]:
# B: grading prompt variants, screening run on the first N_SCREEN items.
#
# Deliberately a screen, not a measurement. We are looking for a large effect --
# accuracy moving off 0.500 -- which n=100 can detect. A variant that only
# nudges the number is not interesting enough to justify a full 300-item run,
# and reading a small difference here as real is exactly the mistake the
# reference run's confidence intervals were added to prevent.
import pilot.parsing

VARIANTS_TO_TEST = ["baseline", "restate", "balanced", "confidence"]
K_VARIANT = 5

variant_path = (f"{CHECKPOINT_DIR}/grading_variants_"
                f"{MODEL_ID.split('/')[-1]}_n{N_SCREEN}_seed{SEED}.jsonl")

variant_results = []
if os.path.exists(variant_path):
    with open(variant_path) as f:
        variant_results = [json.loads(line) for line in f if line.strip()]
    print(f"Resuming from {len(variant_results)} completed (item, variant) pairs")

done = {(r["item_idx"], r["variant"]) for r in variant_results}
todo = [(i, v) for i in range(N_SCREEN) for v in VARIANTS_TO_TEST
        if (i, v) not in done]

if not todo:
    print("All variant runs already done.")
else:
    with tqdm(total=len(todo) * K_VARIANT, desc="grading variants", unit="sample") as pbar:
        for item_idx, variant in todo:
            item = sample[item_idx]
            messages = pilot.prompts.build_grading_messages_variant(item["image"], variant)
            try:
                texts, _, _ = generate_with_logprobs(messages, K_VARIANT, TEMP)
            except ImageTooLargeForGPU:
                print(f"  item {item_idx} / {variant}: image too large; skipping.",
                      flush=True)
                texts = []
            row = {
                "item_idx": item_idx,
                "variant": variant,
                "has_error": bool(item["has_error"]),
                "orig_q": item["orig_q"],
                "pert_a": item["pert_a"],
                "samples_raw": texts,
            }
            variant_results.append(row)
            with open(variant_path, "a") as f:
                f.write(json.dumps(row, default=str) + "\n")
                f.flush()
            pbar.update(K_VARIANT)

print(f"variant_results: {len(variant_results)} (item, variant) pairs")

grading variants:   0%|          | 0/2000 [00:00<?, ?sample/s]

variant_results: 400 (item, variant) pairs


In [15]:
# Code refresh cell -- run after the sampling loop, before scoring.
#
# Cell 2 cloned the repo when the session started. Analysis code (the
# pre-registered thresholds, scoring helpers) may have moved since, and the
# scoring cell below must run the current version. This pulls and reloads in
# dependency order, so you do not have to re-run cell 2 -- which would rm -rf
# the clone, re-mount Drive and re-prompt for tokens.
#
# Safe at this point: `sample`, `raw_results` and the loaded model all stay in
# memory. Only repo/ and module state are touched.
import importlib
import os
import shutil
import subprocess
import sys


def _run(*args):
    result = subprocess.run(args, capture_output=True, text=True)
    text = ((result.stdout or "") + (result.stderr or "")).strip()
    if text:
        print(text[:500])
    return result.returncode


if _run("git", "-C", "repo", "pull", "--ff-only") != 0:
    # A dirty or diverged clone can't fast-forward; a fresh clone always can.
    print("Fast-forward pull failed -- re-cloning.")
    if "REPO_URL" not in globals():
        raise RuntimeError(
            "REPO_URL is not defined -- the runtime was restarted. "
            "Re-run cell 2 (auth & clone) before this cell."
        )
    shutil.rmtree("repo", ignore_errors=True)
    _run("git", "clone", "-q", REPO_URL, "repo")

REPO_DIR = os.path.abspath("repo")
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
importlib.invalidate_caches()

# Order matters: canonicalize imports names from entropy, so entropy has to be
# reloaded first or canonicalize would rebind to the stale objects.
import pilot
import pilot.canonicalize
import pilot.data
import pilot.entropy
import pilot.parsing
import pilot.plotting
import pilot.semantic

for _module in (pilot.entropy, pilot.canonicalize, pilot.parsing,
                pilot.semantic, pilot.plotting, pilot.data):
    importlib.reload(_module)

# Fail loudly here rather than with a confusing AttributeError mid-scoring.
assert hasattr(pilot.canonicalize, "canonical_answer_label"), (
    "Refresh did not pick up canonical_answer_label -- the clone is stale. "
    "Re-run cell 2 to force a clean clone."
)
print("pilot modules reloaded from", os.path.dirname(pilot.__file__))
print("pre-registration registered:",
      pilot.plotting.SCALEUP_PREREGISTRATION["registered"])

# Canonicalization silently degrades without SymPy's LaTeX parser, which needs
# a matching antlr4 runtime. When it is missing, every answer falls back to
# plain-text matching: entropy comes out higher, accuracy lower, and nothing
# errors. Check it explicitly and install if needed, before scoring runs.
if not pilot.canonicalize.latex_parser_available():
    print("SymPy LaTeX parser unavailable -- installing the pinned antlr4 runtime...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "antlr4-python3-runtime==4.11"], check=False)
    importlib.reload(pilot.canonicalize)
print("SymPy LaTeX parser available:",
      pilot.canonicalize.warn_if_latex_parser_missing())

# The optional NLI cell at the end needs sentence-transformers. Installing it
# here is safe -- generation is finished, so a transformers version change can
# no longer disturb the model that produced the samples.
try:
    import sentence_transformers  # noqa: F401
    print("sentence-transformers already present")
except ImportError:
    print("installing sentence-transformers for the optional NLI cell...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "sentence-transformers"], check=False)

Already up to date.
pilot modules reloaded from /content/repo/pilot
pre-registration registered: 2026-08-02
SymPy LaTeX parser available: True
sentence-transformers already present


In [16]:
# The reference run's CSV, needed to join the new samples onto the old ones.
# Pulled from Drive rather than the repo, since results/ is never committed.
REFERENCE_CSV = ("/content/drive/MyDrive/uncertainty-math-vlm/results/"
                 "scaleup_n300_bal50_qwen25-vl-3b-instruct_20260802T163202Z.csv")
assert os.path.exists(REFERENCE_CSV), (
    f"Reference CSV not found at {REFERENCE_CSV}. It is the 2026-08-02 n=300 run; "
    "without it the new samples cannot be joined to the originals."
)
print("reference run found:", os.path.basename(REFERENCE_CSV))

reference run found: scaleup_n300_bal50_qwen25-vl-3b-instruct_20260802T163202Z.csv


In [17]:
# Scoring. Three separate questions, kept separate.
import importlib

import numpy as np
import pandas as pd

import pilot.canonicalize
import pilot.entropy
import pilot.parsing
import pilot.plotting

for m in (pilot.parsing, pilot.canonicalize, pilot.entropy, pilot.plotting):
    importlib.reload(m)

# ---- B: did any prompt variant move grading accuracy off chance? ----
rows = []
for r in variant_results:
    if not r["samples_raw"]:
        continue
    digits = [pilot.parsing.parse_grading(t) for t in r["samples_raw"]]
    confs = [pilot.parsing.parse_grading_confidence(t) for t in r["samples_raw"]]
    labels = [None if d is None else str(d) for d in digits]
    majority, _ = pilot.entropy.majority_cluster(labels)
    said_error = majority == "1"
    rows.append({
        "item_idx": r["item_idx"],
        "variant": r["variant"],
        "has_error": r["has_error"],
        "grading_correct": majority in {"0", "1"} and (said_error == r["has_error"]),
        "said_error": said_error,
        "reasoning_entropy": pilot.entropy.cluster_entropy(labels),
        "mean_confidence": float(np.mean([c for c in confs if c is not None]))
                           if any(c is not None for c in confs) else np.nan,
    })
variants_df = pd.DataFrame(rows)

print("B. GRADING PROMPT VARIANTS (screening, n={} items)".format(N_SCREEN))
print(f"{'variant':>12s} {'accuracy':>9s} {'says error':>11s} {'baseline':>9s}")
for v, g in variants_df.groupby("variant"):
    print(f"{v:>12s} {g['grading_correct'].mean():9.3f} "
          f"{g['said_error'].mean():11.1%} {max(g['has_error'].mean(), 1 - g['has_error'].mean()):9.3f}")
print("\n  A variant only earns a full 300-item run if accuracy clears the")
print("  baseline by a wide margin. Small moves here are noise at n=100.")

# ---- A + C: token confidence, and entropy at K=10 ----
ref = pd.read_csv(REFERENCE_CSV)
ref_by_key = {(r["orig_q"], r["pert_a"]): r for _, r in ref.iterrows()}

import ast

combined = []
n_skipped_join = 0
for entry in extra_results:
    if entry.get("skipped_oom") or not entry["samples_raw"]:
        n_skipped_join += 1
        continue
    key = (entry["item"]["orig_q"], entry["item"]["pert_a"])
    base = ref_by_key.get(key)
    if base is None:
        continue
    base_raw = ast.literal_eval(base["all_transcription_samples_raw"])
    all_raw = list(base_raw) + list(entry["samples_raw"])

    labels_k5 = [pilot.canonicalize.canonical_answer_label(pilot.parsing.parse_transcription(t))
                 for t in base_raw]
    labels_k10 = [pilot.canonicalize.canonical_answer_label(pilot.parsing.parse_transcription(t))
                  for t in all_raw]
    majority_k10, _ = pilot.entropy.majority_cluster(labels_k10)
    gt = pilot.canonicalize.canonical_answer_label(base["pert_a"])

    means = [m for m in entry.get("mean_logprob") or [] if m is not None]
    mins = [m for m in entry.get("min_logprob") or [] if m is not None]
    combined.append({
        "orig_q": key[0], "pert_a": key[1],
        "has_error": base["has_error"],
        "perception_entropy_k5": base["perception_entropy"],
        "perception_entropy_k10": pilot.entropy.cluster_entropy(labels_k10),
        "transcription_correct_k5": bool(base["transcription_correct"]),
        "transcription_correct_k10": majority_k10 == gt,
        # Higher log-probability means more confident, so negate to make it an
        # uncertainty score directed the same way as entropy.
        "neg_mean_logprob": -float(np.mean(means)) if means else np.nan,
        "neg_min_logprob": -float(np.mean(mins)) if mins else np.nan,
    })
combined_df = pd.DataFrame(combined)
print(f"\njoined {len(combined_df)}/{len(ref)} items against the reference run")
if n_skipped_join:
    print(f"  {n_skipped_join} item(s) excluded: image too large to vision-encode "
          "on this GPU.")
    print("  Their K=5 reference numbers are unaffected; only the K=10 and")
    print("  token-confidence columns are missing for them.")

have_logprobs = combined_df["neg_mean_logprob"].notna().any()
if not have_logprobs:
    print("\nA. TOKEN CONFIDENCE -- not measured this run (CAPTURE_LOGPROBS is off;")
    print("   output_scores makes the vision tower materialize full attention and")
    print("   OOMs on this GPU for every item). Needs a larger card.")
else:
    print("\nA. TOKEN CONFIDENCE vs ENTROPY (same items, same correctness labels)")
    for col, name in [("perception_entropy_k5", "entropy K=5"),
                      ("perception_entropy_k10", "entropy K=10"),
                      ("neg_mean_logprob", "-mean logprob"),
                      ("neg_min_logprob", "-min logprob")]:
        r = pilot.plotting.bootstrap_auroc_ci(combined_df, col, "transcription_correct_k5")
        print(f"  {name:16s} AUROC {r['auroc']:.3f} [{r['ci_low']:.3f}, {r['ci_high']:.3f}]")

    d = pilot.plotting.bootstrap_auroc_difference_ci(
        combined_df, "perception_entropy_k5", "transcription_correct_k5",
        "neg_mean_logprob", "transcription_correct_k5")
    print(f"\n  paired (entropy - token confidence): {d['difference']:+.3f} "
          f"[{d['ci_low']:+.3f}, {d['ci_high']:+.3f}]  resolved={d['difference_excludes_zero']}")

print("\nC. DOES K=10 SHARPEN THE SIGNAL?")
for col, lab in [("perception_entropy_k5", "K=5"), ("perception_entropy_k10", "K=10")]:
    r = pilot.plotting.bootstrap_auroc_ci(combined_df, col, "transcription_correct_k10")
    n_levels = combined_df[col].round(6).nunique()
    print(f"  {lab:5s} AUROC {r['auroc']:.3f} [{r['ci_low']:.3f}, {r['ci_high']:.3f}]  "
          f"({n_levels} distinct entropy values)")
d = pilot.plotting.bootstrap_auroc_difference_ci(
    combined_df, "perception_entropy_k10", "transcription_correct_k10",
    "perception_entropy_k5", "transcription_correct_k10")
print(f"  paired (K=10 - K=5): {d['difference']:+.3f} "
      f"[{d['ci_low']:+.3f}, {d['ci_high']:+.3f}]  resolved={d['difference_excludes_zero']}")

B. GRADING PROMPT VARIANTS (screening, n=100 items)
     variant  accuracy  says error  baseline
    balanced     0.540       83.0%     0.510
    baseline     0.510       94.0%     0.510
  confidence     0.510       96.0%     0.510
     restate     0.550       38.0%     0.510

  A variant only earns a full 300-item run if accuracy clears the
  baseline by a wide margin. Small moves here are noise at n=100.

joined 300/300 items against the reference run

A. TOKEN CONFIDENCE vs ENTROPY (same items, same correctness labels)
  entropy K=5      AUROC 0.835 [0.787, 0.879]
  entropy K=10     AUROC 0.791 [0.738, 0.843]
  -mean logprob    AUROC 0.537 [0.469, 0.605]
  -min logprob     AUROC 0.460 [0.392, 0.527]

  paired (entropy - token confidence): +0.297 [+0.214, +0.380]  resolved=True

C. DOES K=10 SHARPEN THE SIGNAL?
  K=5   AUROC 0.761 [0.706, 0.813]  (7 distinct entropy values)
  K=10  AUROC 0.806 [0.756, 0.854]  (34 distinct entropy values)
  paired (K=10 - K=5): +0.045 [+0.005, +0.087]

In [18]:
# Save both frames to Drive first, then the repo.
import subprocess
from datetime import datetime, timezone
from getpass import getpass

timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
model_slug = MODEL_ID.split("/")[-1].lower().replace(".", "")
drive_results = "/content/drive/MyDrive/uncertainty-math-vlm/results"
os.makedirs(drive_results, exist_ok=True)
os.makedirs("repo/results", exist_ok=True)

written = []
for df, stem in ((combined_df, f"confidence_k10_{model_slug}_{timestamp}"),
                 (variants_df, f"grading_variants_{model_slug}_{timestamp}")):
    name = f"{stem}.csv"
    df.to_csv(f"{drive_results}/{name}", index=False)
    df.to_csv(f"repo/results/{name}", index=False)
    written.append(name)
    print(f"wrote {name} ({len(df)} rows)")

_REDACT = []


def git(*args):
    result = subprocess.run(["git", "-C", "repo", *args], capture_output=True, text=True)
    output = (result.stdout or "") + (result.stderr or "")
    for secret in _REDACT:
        if secret:
            output = output.replace(secret, "***")
    if result.returncode != 0 and output.strip():
        print(output.strip())
    return result


git("config", "user.email", "colab-pilot@localhost")
git("config", "user.name", "Colab Pilot Run")
for name in written:
    git("add", f"results/{name}")
git("commit", "-m", f"Add confidence/K=10 and grading-variant results: {timestamp}")

GH_PUSH_TOKEN = (globals().get("GH_TOKEN") or "").strip()
if not GH_PUSH_TOKEN:
    GH_PUSH_TOKEN = getpass("GitHub token (to push results), then press Enter: ").strip()
_REDACT.append(GH_PUSH_TOKEN)

if not GH_PUSH_TOKEN:
    print("No token given -- skipping push. CSVs are safe on Drive.")
else:
    push_url = REPO_URL.replace("https://", f"https://{GH_PUSH_TOKEN}@")
    if git("fetch", push_url, "main").returncode == 0:
        if git("rebase", "FETCH_HEAD").returncode != 0:
            git("rebase", "--abort")
    if git("push", push_url, "HEAD:main").returncode == 0:
        print("Pushed results.")
    else:
        print("Push failed. CSVs are safe on Drive and in repo/results/.")

wrote confidence_k10_qwen25-vl-3b-instruct_20260804T203938Z.csv (300 rows)
wrote grading_variants_qwen25-vl-3b-instruct_20260804T203938Z.csv (400 rows)
remote: Permission to sepehrmaleki369/uncertainty-math-vlm.git denied to sepehrmaleki369.
fatal: unable to access 'https://github.com/sepehrmaleki369/uncertainty-math-vlm.git/': The requested URL returned error: 403
Push failed. CSVs are safe on Drive and in repo/results/.
